Trying with a truth vector

In [ ]:
import torch
import pyvene as pv
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM


model_dir = Path("../.cache/models/")

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=model_dir)
model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir=model_dir, torch_dtype=torch.bfloat16, trust_remote_code=True)

def pv_patcher(b, s): return b + s*0.1

config = pv.IntervenableConfig({
    "layer": 11,
    "component": "mlp_output",
    "intervention": pv_patcher
})

pv_model = pv.IntervenableModel(config, model=model)

qa_prompt = "In which decade did Billboard magazine first publish and American hit chart?"
prompt = tokenizer(qa_prompt, return_tensors="pt")
source_prompt = "in the 1930s" # correct answer
truth_vector = tokenizer(source_prompt, return_tensors="pt")
_, intervened_story = pv_model.generate(
    prompt, truth_vector,
    unit_locations = {"sources->base": 0},
    max_new_tokens=20
)

print(tokenizer.decode(
    intervened_story[0], 
    skip_special_tokens=True
))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In which decade did Billboard magazine first publish and American hit chart?

The answer is, in the early 1990s, the magazine was publishing a lot of hits


Trying with adding noise and scale value

In [ ]:
import torch
import pyvene as pv
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

model_dir = Path("../.cache/models/")

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=model_dir)
model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir=model_dir, torch_dtype=torch.bfloat16, trust_remote_code=True)

qa_prompt = "In which decade did Billboard magazine first publish and American hit chart?"
base_inputs = tokenizer(qa_prompt, return_tensors="pt")

config = pv.IntervenableConfig({
    "layer": 11,
    "component": "mlp_output",
    "intervention_type": pv.AdditionIntervention
})

intervenable_model = pv.IntervenableModel(config, model=model)

noise_vector = torch.rand(model.config.n_embd) * 0.5  # random noise

target_token_index = base_inputs.input_ids.shape[1] - 1

_, intervened_outputs = intervenable_model.generate(
    base=base_inputs,
    unit_locations={"base": 0},
    source_representations=noise_vector,
    max_new_tokens=20
)

print("Intervened Output:", tokenizer.decode(intervened_outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Intervened Output: In which decade did Billboard magazine first publish and American hit chart?

"I think it was in the early '90s, and I think it was in


In [19]:
# unintervened
unintervened_text, _ = intervenable_model.generate(
    base=base_inputs,
    unit_locations={"base": 0},
    source_representations=None,
    max_new_tokens=20,
    output_original_output=True
)

# add noise
_, intervened_text = intervenable_model.generate(
    base=base_inputs,
    unit_locations={"base": 0},
    source_representations=noise_vector,
    max_new_tokens=20
)

print(f"Input prompt: {qa_prompt}")
print(f"Unintervened output:\n{tokenizer.decode(unintervened_text[0], skip_special_tokens=True)}")
print(f"Intervened output:\n{tokenizer.decode(intervened_text[0], skip_special_tokens=True)}")

print("="*60)

for scale in [0.0, 1.0, 2.0, 5.0]:
    _, output = intervenable_model.generate(
        base=base_inputs,
        unit_locations={"base": 0},
        source_representations=noise_vector * scale,
        max_new_tokens=50
    )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"Scale {scale}: {text}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Input prompt: In which decade did Billboard magazine first publish and American hit chart?
Unintervened output:
In which decade did Billboard magazine first publish and American hit chart?

I think it was in the early '80s. I think it was in the '
Intervened output:
In which decade did Billboard magazine first publish and American hit chart?

"I think it was in the early '90s, and I think it was in


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Scale 0.0: In which decade did Billboard magazine first publish and American hit chart?

I think it was in the early '80s. I think it was in the '80s. I think it was in the '80s. I think it was in the '80s. I think it was in the '


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Scale 1.0: In which decade did Billboard magazine first publish and American hit chart?

"I think it was in the early '90s, and I think it was in the '90s, and I think it was in the '90s, and I think it was in the '90s, and I think


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Scale 2.0: In which decade did Billboard magazine first publish and American hit chart?

The answer is, in the early 1990s, the magazine was publishing a number of books, including the first two books of the Beatles' "The Beatles" and the first two albums of the Rolling Stones' "The Rolling Stones."

Scale 5.0: In which decade did Billboard magazine first publish and American hit chart?

"I think it was in the early '80s, and I think it was in the '90s," he says. "I think it was in the '90s, and I think it was in the '90s,


Trying with addition intervention and steering vector = correct activation - incorrect activation

In [3]:
import torch
import pyvene as pv
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

model_dir = Path("../.cache/models/")

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=model_dir)
model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir=model_dir, torch_dtype=torch.bfloat16, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

base_qa = "In which decade did Billboard magazine first publish and American hit chart?"
pos_prompt = "It is in the early 1930s" # correct answer
neg_prompt = "It is in the early 1980s"

def get_last_layer_activation(prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        return outputs.hidden_states[-1][0, -1, :].unsqueeze(0)

# Steering Vector
pos_activation = get_last_layer_activation(pos_prompt)
neg_activation = get_last_layer_activation(neg_prompt)
steering_vector = pos_activation - neg_activation

config = pv.IntervenableConfig({
    "layer": 11,  # last layer
    "component": "mlp_output",
    "intervention_type": pv.AdditionIntervention
})
intervenable_model = pv.IntervenableModel(config, model=model)

base_inputs = tokenizer(base_qa, return_tensors="pt")
target_idx = base_inputs.input_ids.shape[1]

steering_strength = 2
scaled_vector = steering_vector * steering_strength

_, generated_outputs = intervenable_model.generate(
    base=base_inputs,
    unit_locations={"base": 0},
    source_representations=scaled_vector,
    max_new_tokens=20,
    # use_cache=False,
    intervene_on_prompt=True,
    do_sample=False
)

print(f"Input Prompt: {base_qa}")
print(f"Intervened Output: {tokenizer.decode(generated_outputs[0], skip_special_tokens=True)}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Input Prompt: In which decade did Billboard magazine first publish and American hit chart?
Intervened Output: In which decade did Billboard magazine first publish and American hit chart?

I think it was in the early '80s. I think it was in the '


Steering with a customized patcher (still correct activation - incorrect activation)

In [8]:
import torch
import pyvene as pv
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

model_dir = Path("../.cache/models/")

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=model_dir)
model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir=model_dir, torch_dtype=torch.bfloat16, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

base_qa = "In which decade did Billboard magazine first publish and American hit chart?"
pos_prompt = "It is in the early 1930s" # correct answer
neg_prompt = "It is in the early 1980s"

def get_last_layer_activation(prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        return outputs.hidden_states[-1][0, -1, :].unsqueeze(0)

# Steering Vector
pos_activation = get_last_layer_activation(pos_prompt)
neg_activation = get_last_layer_activation(neg_prompt)
steering_vector = pos_activation - neg_activation

class SteeringIntervention(pv.ConstantSourceIntervention):
    def __init__(self, **kwargs):
        super().__init__(**kwargs, keep_last_dim=True) # you must set keep_last_dim=True to get tokenized reprs.
        self.called_counter = 0

    def forward(self, base, source=None, subspaces=None, **kwargs):
        if subspaces["logging"]:
            print(f"(called {self.called_counter} times) incoming reprs shape:", base.shape)
        self.called_counter += 1
        return base + subspaces["mag"] * steering_vector

# Mount the intervention to our steering model
pv_config = pv.IntervenableConfig(representations=[{
    "layer": 11,
    "component": "mlp_output",
    "low_rank_dimension": 1,
    "intervention": SteeringIntervention(embed_dim=model.config.hidden_size, 
                                      low_rank_dimension=1)}])
pv_model = pv.IntervenableModel(pv_config, model)
# pv_model.set_device("cuda")

In [9]:
prompt = "In which decade did Billboard magazine first publish and American hit chart?"

prompt = tokenizer.decode(tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}], 
    tokenize=True, add_generation_prompt=True)[1:])

inputs = tokenizer(
    prompt, return_tensors="pt", padding=True, truncation=True
)
_, generations = pv_model.generate(
    inputs, 
    unit_locations=None,      # set to None means intervention will be applied for each forward call
    intervene_on_prompt=True, # intervention will be called for the prompt kv cache call
    subspaces=[{"mag": 2.0, "logging": True}], # other metadata
    max_new_tokens=10, do_sample=True, temperature=1.0)

print(f"Input Prompt: {base_qa}")
print(f"Intervened Output: {tokenizer.decode(generated_outputs[0], skip_special_tokens=True)}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


(called 0 times) incoming reprs shape: torch.Size([1, 13, 768])
Input Prompt: In which decade did Billboard magazine first publish and American hit chart?
Intervened Output: In which decade did Billboard magazine first publish and American hit chart?

I think it was in the early '80s. I think it was in the '


In [ ]:
import torch
from pathlib import Path
from transformers import LlamaTokenizer, LlamaForCausalLM

model_dir = Path("../.cache/models/")
model_path = 'openlm-research/open_llama_7b'

tokenizer = LlamaTokenizer.from_pretrained(model_path)
model = LlamaForCausalLM.from_pretrained(
    model_path, cache_dir=model_dir, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True
)

prompt = "In which decade did Billboard magazine first publish and American hit chart?"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids

generation_output = model.generate(
    input_ids=input_ids, max_new_tokens=100
)
print(tokenizer.decode(generation_output[0]))


You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/data/share/project/RSML/llm-hallucinations-factual-qa/.venv/lib/python3.10/site-packages/transformers/generation/utils.py:1510: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


<s>In which decade did Billboard magazine first publish and American hit chart?
In which decade did Billboard magazine first publish and American hit chart? The Billboard Hot 100 is a chart that ranks the top-performing singles of the United States. It is published weekly by Billboard magazine. The data are compiled by Nielsen SoundScan based collectively on each single's weekly physical and digital sales, as well as airplay.
The Billboard Hot 100 is a chart that ranks the top-performing singles of the United States. It is published


In [18]:
prompt = "Which was the first European country to abolish capital punishment?"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids

generation_output = model.generate(
    input_ids=input_ids, max_new_tokens=30, do_sample=False
)
print(tokenizer.decode(generation_output[0]))

<s>Which was the first European country to abolish capital punishment?
Which was the first European country to abolish capital punishment? The first European country to abolish capital punishment was France in 179


In [2]:
import torch
import pyvene as pv
from pathlib import Path
from transformers import LlamaTokenizer, LlamaForCausalLM

model_dir = Path("../.cache/models/")

model_name = "openlm-research/open_llama_7b"
tokenizer = LlamaTokenizer.from_pretrained(model_name, cache_dir=model_dir)
model = LlamaForCausalLM.from_pretrained(model_name, cache_dir=model_dir, torch_dtype=torch.bfloat16, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

base_qa = "Which was the first European country to abolish capital punishment?"
pos_prompt = "It was Norway" # correct answer
neg_prompt = "It was France"

def get_last_layer_activation(prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        return outputs.hidden_states[-1][0, -1, :].unsqueeze(0)

# Steering Vector
pos_activation = get_last_layer_activation(pos_prompt)
neg_activation = get_last_layer_activation(neg_prompt)
steering_vector = pos_activation - neg_activation

config = pv.IntervenableConfig({
    "layer": 31,  # last layer
    "component": "mlp_output",
    "intervention_type": pv.AdditionIntervention
})
intervenable_model = pv.IntervenableModel(config, model=model)

base_inputs = tokenizer(base_qa, return_tensors="pt")
target_idx = base_inputs.input_ids.shape[1]

steering_strength = 1.5
scaled_vector = steering_vector * steering_strength

_, generated_outputs = intervenable_model.generate(
    base=base_inputs,
    unit_locations={"base": 0},
    source_representations=scaled_vector,
    max_new_tokens=32,
    intervene_on_prompt=True,
    do_sample=False
)

print(f"Input Prompt: {base_qa}")
print(f"Intervened Output: {tokenizer.decode(generated_outputs[0], skip_special_tokens=True)}")

The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Input Prompt: Which was the first European country to abolish capital punishment?
Intervened Output: Which was the first European country to abolish capital punishment?
Which was the first European country to abolish capital punishment? The first European country to abolish capital punishment was France in 1792.
